# Exploring CryoSwath Diagnostic Hooks

CryoSwath functions can expose low-profile hooks for development, improvement, exploration, introspection, validation, and debugging. This tutorial uses synthetic data to show how to capture intermediate state from hypsometric interpolation without adding plotting code to the production routine.

The important idea is simple: pass a callback as `diagnostic_hook`. CryoSwath calls it at stable stage boundaries with an event name and a payload. When the hook is omitted, the function behaves normally.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from cryoswath.misc import interpolate_hypsometrically
from cryoswath.test_plots import hypsometry

%matplotlib inline

## Create a small synthetic dataset

The synthetic surface has a smooth elevation relationship, deterministic gaps, and one local outlier. This is enough to exercise the same diagnostic stages that are useful in real L3/L4 development.

In [ ]:
ny, nx = 16, 16
y = np.arange(ny)
x = np.arange(nx)
yy, xx = np.meshgrid(y, x, indexing="ij")

ref_elev = xx * 40 + yy * 7.5
surface = 0.012 * ref_elev + 0.000015 * ref_elev**2
surface = surface + np.sin(xx / 2) * 0.05
surface = surface.astype(float)
error = np.ones_like(surface) * 0.5

missing = (xx + yy) % 7 == 0
surface[missing] = np.nan
error[missing] = np.nan
surface[8, 8] += 10

ds = xr.Dataset(
    {
        "h": (("y", "x"), surface),
        "h_std": (("y", "x"), error),
        "ref_elev": (("y", "x"), ref_elev),
    },
    coords={"x": x, "y": y},
).stack(stacked_x_y=["x", "y"])

ds

## Run without diagnostics

The default call fills the data and emits nothing. This is the production behavior: no global debug state, no mandatory plotting dependency, and no side effects.

In [ ]:
filled_default = interpolate_hypsometrically(ds, "h", "h_std")

int(ds.h.isnull().sum()), int(filled_default.h.isnull().sum())

## Capture diagnostic events in memory

A hook is just a function that accepts `(name, payload)`. For exploration, collecting events in a list is often enough.

In [ ]:
events = []


def capture_event(name, payload):
    events.append((name, payload))


filled = interpolate_hypsometrically(
    ds,
    "h",
    "h_std",
    outlier_replace=True,
    diagnostic_hook=capture_event,
)

[name for name, _payload in events]

Each payload contains objects that already exist inside the algorithm at that stage. You can inspect them directly before deciding whether a quick-look plot, a report, or a saved artifact is useful.

In [ ]:
event_name, payload = events[-1]
event_name, sorted(payload)

## Plot the captured payloads

`cryoswath.test_plots.hypsometry` knows how to visualize these payloads. This keeps matplotlib out of the numerical routine while still making the internal state easy to inspect.

In [ ]:
for name, payload in events:
    hypsometry.plot_event(name, payload)
    plt.show()

## Optionally write selected artifacts with `snippets`

The `lab` environment installs `snippets`, which provides a small `Diagnostics` helper for writing selected artifacts. The import is optional here so this tutorial still runs in lean test environments.

In [ ]:
try:
    from snippets.debugging import Diagnostics
except ImportError:
    Diagnostics = None


def diagnostics_writer(diagnostics):
    def emit(name, payload):
        def write(path: Path):
            result = hypsometry.plot_event(name, payload)
            fig = result if hasattr(result, "savefig") else result.figure
            fig.savefig(path / "plot.png", dpi=150, bbox_inches="tight")
            plt.close(fig)

        diagnostics.emit(name, write)

    return emit


if Diagnostics is None:
    print("Install cryoswath[lab] or use `pixi shell -e lab` for snippets.")
else:
    diagnostics = Diagnostics(
        enabled=True,
        outdir="debug/hypsometry",
        only={"hypsometry.fit_fill_mask"},
    )
    _ = interpolate_hypsometrically(
        ds,
        "h",
        "h_std",
        outlier_replace=True,
        diagnostic_hook=diagnostics_writer(diagnostics),
    )
    print("Wrote selected diagnostics to debug/hypsometry")

The same pattern can be used in scripts that run larger CryoSwath workflows. Keep hooks at stage boundaries, keep the payloads explicit, and keep plotting or artifact writing outside the production algorithm.